# Analysing existing files

`DatasetDescription.describe_dataset` measures a dataset from its file: dimensions, sites,
occupied sites, density, overlap with the first variable, unused coordinates and dtype. It reads
only the file, so it works on data tabray did not write.

This notebook first generates a small dataset and writes it as netCDF and as parquet. After that
it uses only the two file paths, and compares the two files step by step: sizes, contents,
description, NaN. To analyse other files, set `NC_PATH`, `PQ_PATH` and `COORDS` and skip the
generation cell.

In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr

from tabray.generate_data import GenerateData
from tabray.output import DatasetDescription
from tabray.utils.viz import format_bytes

OUT_DIR = Path("./analyse_dataset")
NC_PATH = OUT_DIR / "netCDF" / "ds.nc"
PQ_PATH = OUT_DIR / "parquet"  # directory of the parquet dataset
COORDS = ["x0", "x1", "x2"]  # parquet has no axes: name the coordinate columns

## 1. Generate a small dataset

Three variables on a 6x6x3 grid: var0 and var1 vary along all three dimensions, var2 along two
and is pinned to one `x0`. The density of var0 is a share of the grid, so the validation corrects
`num_obs` from 48 to 32.

In [ ]:
gen = GenerateData(
    num_obs=48,
    num_dims=3,
    ratio_dims=[2, 2, 1],
    density=[0.3, 0.15, 0.3],
    seed=2026,
    num_vars=3,
    var_dims=[3, 3, 2],
    overlap=[0.5, 0.5],
)
gen.generate(
    netcdf_filepath=str(NC_PATH),
    parquet_filepath=str(PQ_PATH / "ds"),
)

From here on only `NC_PATH` and `PQ_PATH` are used.

## 2. Open both files

In [ ]:
nc = xr.open_dataset(NC_PATH)
pq = pd.read_parquet(PQ_PATH)
pq_files = sorted(p for p in PQ_PATH.iterdir() if p.is_file())

## 3. Sizes

- netCDF: every coordinate stored once, and every variable its whole grid, vacant sites as NaN
- parquet: one row per site holding at least one variable; each row repeats its coordinates, and
  a variable absent at that site is a NaN cell

In [ ]:
nc_values = sum(nc[name].size for name in nc.data_vars)
nc_coords = sum(nc.sizes.values())
pq_coords = len(pq) * len(COORDS)
pq_values = len(pq) * (len(pq.columns) - len(COORDS))

pd.DataFrame(
    {
        "netCDF": {
            "on disk": format_bytes(NC_PATH.stat().st_size),
            "in memory": format_bytes(nc.nbytes),
            "shape": " x ".join(f"{dim}={n}" for dim, n in nc.sizes.items()),
            "coordinate values": nc_coords,
            "data values": nc_values,
            "stored values": nc_coords + nc_values,
            "NaN": int(sum(nc[name].isnull().sum() for name in nc.data_vars)),
        },
        "parquet": {
            "on disk": format_bytes(sum(p.stat().st_size for p in pq_files)),
            "in memory": format_bytes(pq.memory_usage(index=True, deep=True).sum()),
            "shape": f"{len(pq)} rows x {len(pq.columns)} columns",
            "coordinate values": pq_coords,
            "data values": pq_values,
            "stored values": pq_coords + pq_values,
            "NaN": int(pq.drop(columns=COORDS).isna().to_numpy().sum()),
        },
    }
)

## 4. Contents

In [ ]:
nc

In [ ]:
pq.head(10)

## 5. Description, one variable at a time

Parquet has no axes, so `describe_dataset` infers the grid from the coordinate values that
appear, and a variable's dimensions from the coordinate columns along which it takes more than
one value. On other data the two can differ for that reason: a coordinate no row uses is missing
from the parquet grid, and a variable with a single value along an axis looks constant along it.

In [ ]:
nc_table = DatasetDescription.describe_dataset(str(NC_PATH))
pq_table = DatasetDescription.describe_dataset(str(PQ_PATH), coords=COORDS)

print("grid  netCDF: ", nc_table.attrs["grid"])
print("grid  parquet:", pq_table.attrs["grid"])

# rows var0/netCDF, var0/parquet, var1/netCDF, ...
both = pd.concat({"netCDF": nc_table, "parquet": pq_table}).swaplevel()
both.loc[nc_table.index]

In [ ]:
columns = ["dims", "var_sites", "var_sites_occupied", "density", "overlap", "unused_coords"]
pd.testing.assert_frame_equal(nc_table[columns], pq_table[columns])
print("netCDF and parquet descriptions agree on", ", ".join(columns))

## 6. NaN per variable

The netCDF stores NaN at every site of a variable's grid it does not hold; the parquet stores NaN
only at rows where another variable holds the site.

In [ ]:
pd.DataFrame(
    {
        "netCDF values": {name: nc[name].size for name in nc.data_vars},
        "netCDF NaN": {name: int(nc[name].isnull().sum()) for name in nc.data_vars},
        "parquet values": {name: len(pq) for name in nc.data_vars},
        "parquet NaN": {name: int(pq[name].isna().sum()) for name in nc.data_vars},
    }
)

In [ ]:
nc.close()